In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 86,
 'tn': 2600,
 'fp': 37,
 'fn': 277,
 'misclassification_rate': 0.10466666666666667,
 'false_positive_rate': 0.014031095942358742,
 'false_negative_rate': 0.7630853994490359}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 25,
 'tn': 867,
 'fp': 7,
 'fn': 101,
 'misclassification_rate': 0.108,
 'false_positive_rate': 0.008009153318077803,
 'false_negative_rate': 0.8015873015873016}

#### misclassification rate before changing anything = 0.136

In [10]:
#after changing n_estimator = 100 -> misclassification rate = 0.139
#after changing n_estimator = 200 -> misclassification rate = 0.131
#after changing n_estimator = 300 -> misclassification rate = 0.131

#after changing learning_rate = 0.05 -> misclassification rate = 0.125
#after changing learning_rate = 0.03 -> misclassification rate = 0.126

#after changing max_depth = 4 -> 0.12
#after changing max_depth = 3 -> 0.117
#after changing max_depth = 2 -> 0.111
#after changing max_depth = 2 -> 0.109

# after changing subsample=0.8, 0.108


# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

#### Based on the final misclassification rate of 0.108 on the test set, we can correctly classify about 89.2% of unseen data, which makes me relatively confident in the model's ability to predict a bot. This is a meaningful improvement from the prior misclassifiation rate of 0.139. 

### What are potential ramifications of false positives from the model?

#### False positives result in actual human users being incorrectly flagged as bots, which in a real-life setting may lead to unnecessary account restrictions, which could negatively affect user experience.

### What are potential ramifications of false negatives from the model?

#### False negatives result in bots being incorrectly classifified as actual human users, which in a real-life setting could be a security concern.